# Doing economics with python

This notebook contains code from: https://janboone.github.io/python_economics/economics.html

Course: Applied Economic Analysis at Tilburg University

In [ ]:
import pandas as pd
import numpy as np
import pymc3 as pm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
import random
import wbdata as wb

plt.style.use('seaborn')
%matplotlib inline


In [ ]:
number_of_agents = 1000
number_of_goods = 100

valuations = sorted(pm.Normal.dist(100,20).random(size=number_of_agents),reverse = True)


In [ ]:
print("{0:.2f}".format(valuations[number_of_goods]))


In [ ]:
def excess_demand(p,valuations,number_of_goods):
    return demand(p,valuations)-number_of_goods


In [ ]:
price = optimize.fsolve(lambda x: excess_demand(x,valuations,number_of_goods),120)
print("{0:.2f}".format(price))


In [ ]:
"{0:.2f}".format(max_welfare)


In [ ]:
random.shuffle(valuations)
valuations_supply = valuations[:number_of_goods]
valuations_demand = valuations[number_of_goods:]


In [ ]:
incomes = pm.Normal.dist(100,20).random(size=number_of_agents)


In [ ]:
def afford(p,incomes):
    return incomes>p

def wtp(p,valuations):
    return valuations>p

def demand_2(p,valuations,incomes):
    return np.sum(afford(p,incomes)*wtp(p,valuations))


In [ ]:
def profit(p,valuations):
    return p*demand(p,valuations)


In [ ]:
range_p = np.arange(0,140)

plt.plot(range_p, [profit(p,valuations) for p in range_p], label = "profit")
plt.legend()
plt.xlabel("$P$")
plt.ylabel("$\pi$")
plt.show()


In [ ]:
price


In [ ]:
c0 = 0.3
vector_c = [c0]*3

def p(Q):
    return 1 - Q

def costs(q,c):
    return c*q

def profits(q,Q_other,c):
    return p(q+Q_other)*q-costs(q,c)

def reaction(Q_other,c):
    q1 =  optimize.fminbound(lambda x: -profits(x,Q_other,c),0,1,full_output=1)
    return q1[0]

def fixed_point_three_firms(vector_q,vector_c):
    return [vector_q[0]-reaction(vector_q[1]+vector_q[2],vector_c[0]),
            vector_q[1]-reaction(vector_q[0]+vector_q[2],vector_c[1]),
            vector_q[2]-reaction(vector_q[0]+vector_q[1],vector_c[2])]


In [ ]:
initial_guess_3 = [0,0,0]

Q0 = np.sum(optimize.fsolve(lambda q: fixed_point_three_firms(q,vector_c), initial_guess_3))
P0 = p(Q0)
H0 = 3*(1.0/3.0)**2

print("Before the merger")
print("=================")
print("total output: {:.3f}".format(Q0))
print("equil. price: {:.3f}".format(P0))
print("Herfn. index: {:.3f}".format(H0))


In [ ]:
c_after_merger = pm.Uniform.dist(0,c0).random(size = 100)

initial_guess = [0.2,0.2]

q1_after_merger = [optimize.fsolve(lambda q: fixed_point_two_firms(q,[c0,c]), initial_guess)[0] for c in c_after_merger]
q2_after_merger = [optimize.fsolve(lambda q: fixed_point_two_firms(q,[c0,c]), initial_guess)[1] for c in c_after_merger]


In [ ]:
np.sum(df_after_merger.P < P0)/len(df_after_merger.P)


In [ ]:
number_of_agents = 1000
valuations = np.array(sorted(pm.Normal.dist(100,20).random(size=number_of_agents),reverse = True))

def demand(p,valuations):
    return sum(valuations>p)

c = 30
γ = 80
def costs(q):
    return c*q

def externality(q):
    return γ*q

def profit_c(p,valuations):
    return p*demand(p,valuations)-costs(demand(p,valuations))

def welfare_e(p,valuations):
    return np.sum(valuations[:demand(p,valuations)])-costs(demand(p,valuations))-externality(demand(p,valuations))


In [ ]:
income = 1.1
cost = 1
ρ = 0.1
def u(x):
    return x**ρ

number_of_agents = 50

π = pm.Uniform.dist(0.0,1.0).random(size = number_of_agents)
π.sort()


In [ ]:
def insurance_supply(Q):
    return np.mean(π[-Q:])*cost


In [ ]:
range_Q = np.arange(1,number_of_agents+1,1)
range_sigma = np.arange(0,1.01,0.01)
plt.plot(range_Q,[insurance_supply(Q) for Q in range_Q],label="insurance supply")
plt.plot([insurance_demand(sigma) for sigma in range_sigma],range_sigma,label="insurance demand")
plt.plot(range_Q,[π[-Q]*cost for Q in range_Q],label="marginal cost")
plt.legend()
plt.xlabel('$Q$')
plt.ylabel('$\sigma$')
plt.title('Perfectly competitive insurance market')
plt.show()


In [ ]:
number_of_agents = 200
effort_costs = pm.Lognormal.dist(mu=0.0,sd=0.5).random(size=number_of_agents)
def effort(c,τ):
    sol = optimize.minimize(lambda x: -(x*(1-τ)-c*x**2),1)
    return sol.x


In [ ]:
def Welfare(τ,ρ):
    τ_0 = np.mean([τ*effort(c,τ) for c in effort_costs])
    return (np.sum([((1-τ)*effort(c,τ)+τ_0 - c*effort(c,τ)**2)**ρ for c in effort_costs]))**(1/ρ)


In [ ]:
def profit(x,equity=0):
    return np.mean(np.maximum(x,-equity))


In [ ]:
v_std = np.arange(0,200,1)
v_returns = [pm.Normal.dist(-10,std).random(size=1000) for std in v_std]
plt.scatter([np.std(vx) for vx in v_returns],[profit(vx) for vx in v_returns], label="no equity")
plt.scatter([np.std(vx) for vx in v_returns],[profit(vx,60) for vx in v_returns], label="equity equals 60")
plt.scatter([np.std(vx) for vx in v_returns],[profit(vx,10000) for vx in v_returns], label="social value")
plt.xlabel('$\sigma$')
plt.ylabel('return')
plt.legend()
plt.show()


In [ ]:
y_a = 1
y_g = 10
y_b = -20

def q_g(q,ability):
    return ability*(1-q)*(1+q)

def q_b(q,ability):
    return 1 - q - q_g(q,ability)


In [ ]:
initial_guess = [0.5,1.5]

def bank_choices(ability):
    opt_w = optimize.fmin(lambda x: -(q_g(q_choice(x,ability),ability)*y_g + q_choice(x,ability)*y_a + (1-q_choice(x,ability)-q_g(q_choice(x,ability),ability))*y_b),initial_guess, disp=0)
    return [opt_w,q_choice(opt_w,ability)]

def contract(ability,outside_option):
    q = bank_choices(ability)[1]
    profit = q*y_a + q_g(q,ability)*y_g + q_b(q,ability)*y_b
    if profit - outside_option >= 0:
        q_out = q
    else:
        q_out = -1
    return q_out


In [ ]:
α_l = 0.1
α_h = 0.5

def profit(R,outside_option):
    q = R/(2*α_h)
    w_g = outside_option/(R*q + q_g(q,α_h))
    mimic_q = R/(2*α_l)
    w_a = R*w_g
    wage_l = mimic_q*w_a+q_g(mimic_q,α_l)*w_g
    profit = 0.5*(q*y_a + q_g(q,α_h)*y_g+(1-q-q_g(q,α_h))*y_b - outside_option) - 0.5*wage_l
    return [profit, q]

initial_guess = 0.5

def outcome_h(outside_option):
    wages_h = optimize.fmin(lambda x: -profit(x,outside_option)[0],initial_guess, disp=0)
    return profit(wages_h,outside_option)[1]


In [ ]:
plt.plot(range_outside_options,[outcome_h(o) for o in range_outside_options])
plt.xlabel('outside option')
plt.ylabel('probability of safe choice $q_a$')
plt.show()


In [ ]:
wb.search_indicators("gdp per capita")


In [ ]:
indicators = {"NY.GDP.PCAP.PP.KD": "GDP_per_head"}
df_wb = wb.get_dataframe(indicators, convert_date=True)
df_wb.reset_index(inplace = True)
df_wb.head()


In [ ]:
df_wb.to_csv('data/worldbank_data_gdp_per_capita.csv')


In [ ]:
df_1990=df_wb[df_wb['date']=='1990-01-01']
df_2017=df_wb[df_wb['date']=='2017-01-01']


In [ ]:
df_merged = pd.merge(df_1990, df_2017, on=['country'], suffixes=['_1990', '_2017'], how='inner')


In [ ]:
from bokeh.io import output_file, show, output_notebook
from bokeh.plotting import figure
from bokeh.models import HoverTool
output_notebook()

hover = HoverTool(tooltips=[
     ('country','@country'),
     ])

plot = figure(tools=[hover])
plot.circle('GDP_per_head_1990','GDP_per_head_2017',
    size=10, source=df_merged)
output_file('inequality.html')
show(plot)


In [ ]:
outcomes = ['H','T']

def waiting_time(pattern):
    throws = list(np.random.choice(outcomes,size=3))
    while not check(throws,pattern):
        throws.append(random.choice(outcomes))
    return len(throws)


In [ ]:
mu = 1000
sd = 100
number_of_samples=250

def moments(n):
    samples = pm.Normal.dist(mu,sd).random(size=(number_of_samples,n))
    mus = samples.mean(axis=1)
    std = mus.std()
    return [mus,std]


In [ ]:
def my_function(n):
    return 20


In [ ]:
range_n = np.arange(1,1000)

plt.plot(range_n,[moments(n)[1] for n in range_n], label='moments')
plt.plot(range_n,[my_function(n) for n in range_n], label='my_function')
plt.legend()
plt.xlabel('$n$')
plt.show()


In [ ]:
plt.hist(moments(10)[0],bins=30,label='$n=10$',density=True)
plt.hist(moments(100)[0],bins=30,alpha=0.6,label='$n=100$',density=True)
plt.legend()
plt.show()


In [ ]:
def deductible(c,d):
   return min(c,d)

range_c = np.arange(0,500,0.1)
range_v170 = [deductible(c,170) for c in range_c]
range_v365 = [deductible(c,365) for c in range_c]

plt.plot(range_c,range_v365,'-', color = 'r', linewidth = 2, label = '$d=365$')
plt.plot(range_c,range_v170,'-', color = 'b', linewidth = 2, label = '$d=170$')
plt.legend()
plt.fill_between(range_c, range_v170, range_v365, facecolor='yellow')
plt.ylim(0,500)
plt.xlabel('Cost')
plt.ylabel('Value')
plt.show()


In [ ]:
df_2014 = pd.read_csv('data/Vektis Open Databestand Zorgverzekeringswet 2014 - postcode3.csv', sep = ';')

cost_categories_under_deductible = ['KOSTEN_MEDISCH_SPECIALISTISCHE_ZORG', 'KOSTEN_MONDZORG', 'KOSTEN_FARMACIE', 'KOSTEN_HULPMIDDELEN', 'KOSTEN_PARAMEDISCHE_ZORG_FYSIOTHERAPIE', 'KOSTEN_PARAMEDISCHE_ZORG_OVERIG', 'KOSTEN_ZIEKENVERVOER_ZITTEND', 'KOSTEN_ZIEKENVERVOER_LIGGEND', 'KOSTEN_GRENSOVERSCHRIJDENDE_ZORG', 'KOSTEN_OVERIG']

def get_data_into_shape(df):
    df['health_expenditure_under_deductible'] = df[cost_categories_under_deductible].sum(axis=1)
    df = df.rename({
        'GESLACHT':'sex',
        'LEEFTIJDSKLASSE':'age',
        'GEMEENTENAAM':'MUNICIPALITY',
        'AANTAL_BSN':'number_citizens',
        'KOSTEN_MEDISCH_SPECIALISTISCHE_ZORG':'hospital_care',
        'KOSTEN_FARMACIE':'pharmaceuticals',
        'KOSTEN_TWEEDELIJNS_GGZ':'mental_care',
        'KOSTEN_HUISARTS_INSCHRIJFTARIEF':'GP_capitation',
        'KOSTEN_HUISARTS_CONSULT':'GP_fee_for_service',
        'KOSTEN_HUISARTS_OVERIG':'GP_other',
        'KOSTEN_MONDZORG':'dental care',
        'KOSTEN_PARAMEDISCHE_ZORG_FYSIOTHERAPIE':'physiotherapy',
        'KOSTEN_KRAAMZORG':'maternity_care',
        'KOSTEN_VERLOSKUNDIGE_ZORG':'obstetrics'
    }, axis='columns')
    df.drop(['AANTAL_VERZEKERDEJAREN',
             'KOSTEN_HULPMIDDELEN',
             'KOSTEN_PARAMEDISCHE_ZORG_OVERIG',
             'KOSTEN_ZIEKENVERVOER_ZITTEND',
             'KOSTEN_ZIEKENVERVOER_LIGGEND',
             'KOSTEN_GRENSOVERSCHRIJDENDE_ZORG',
             'KOSTEN_OVERIG',
             'KOSTEN_EERSTELIJNS_ONDERSTEUNING'],inplace=True,axis=1)
    df.drop(df.index[[0]], inplace=True)
    df['sex'] = df['sex'].astype('category')
    df['age'] = df['age'].astype('category')
    df['costs_per_head']=df['health_expenditure_under_deductible']/df['number_citizens']
    df['log_costs_per_head']=np.log(1+df['health_expenditure_under_deductible']/df['number_citizens'])
    df = df[(df['age'] != '90+')]
    df['age'] = df['age'].astype(int)
    return df

df_2014 = get_data_into_shape(df_2014)
df_2014.head()


In [ ]:
costs_per_sex_age = df_2014.groupby(['sex','age'])['costs_per_head'].mean()


In [ ]:
fig = plt.figure()
ax = costs_per_sex_age['M'].plot()
ax = costs_per_sex_age['V'].plot()
ax.set_xlabel('age')
ax.set_ylabel('costs per head')
ax.set_title('average costs per age and sex')
ax.legend(['male','female'])


In [ ]:
df_2014.query('sex=="M" & age=="30"')['log_costs_per_head'].hist(bins=50)


In [ ]:
log_costs_per_age_female = df_2014[df_2014['sex']=='V'].groupby(['age'])['log_costs_per_head'].mean()

log_costs_per_head = df_2014[df_2014['sex']=='V'].log_costs_per_head.values
age = df_2014[df_2014['sex']=='V'].age.values


with pm.Model() as model:
    
    μ = pm.Normal('μ', 8, 3, shape=len(set(age)))
    σ = pm.HalfNormal('σ', 4, shape=len(set(age)))
    z = pm.Normal('z', μ[age], σ[age], observed=log_costs_per_head)


In [ ]:
with model:
    trace = pm.sample(4000,step = pm.Metropolis(),start = pm.find_MAP())


In [ ]:
summary = pm.summary(trace, varnames=['μ'])

pm.plot_posterior(trace, varnames=['μ'],ref_val = log_costs_per_age_female.values)[0]


In [ ]:
plt.plot(summary['mean'].values,label='calculated means')
plt.plot(log_costs_per_age_female,'o',label='observed means')
plt.legend()


In [ ]:
summary['mean']['μ__17'] - summary['mean']['μ__19']


In [ ]:
plt.hist(trace['μ'][:,17],density=True,label='age 17',bins=50)
plt.hist(trace['μ'][:,19],density=True,label='age 19',bins=50)
plt.legend()


In [ ]:
df_2011 = pd.read_csv('data/Vektis Open Databestand Zorgverzekeringswet 2011 - postcode3.csv', sep = ';')

df_2011 = get_data_into_shape(df_2011)
df_2011.head()


In [ ]:
log_costs_per_age_female = df_2011[df_2011['sex']=='V'].groupby(['age'])['log_costs_per_head'].mean()

log_costs_per_head = df_2011[df_2011['sex']=='V'].log_costs_per_head.values
age = df_2011[df_2011['sex']=='V'].age.values


with pm.Model() as model_2011:
    
    μ = pm.Normal('μ', 8, 3, shape=len(set(age)))
    σ = pm.HalfCauchy('σ', 4, shape=len(set(age)))
    z = pm.Normal('z', μ[age], σ[age], observed=log_costs_per_head)


In [ ]:
with model_2011:
    trace_2011 = pm.sample(4000,step = pm.Metropolis(),start = pm.find_MAP())


In [ ]:
summary_2011 = pm.summary(trace_2011, varnames=['μ'])


In [ ]:
summary['mean']['μ__17']-summary['mean']['μ__19']


In [ ]:
summary_2011['mean']['μ__17']-summary_2011['mean']['μ__19']


In [ ]:
plt.hist(trace['μ'][:,17]-trace['μ'][:,19],density=True,label='difference in 2014',bins=50)
plt.hist(trace_2011['μ'][:,17]-trace_2011['μ'][:,19],density=True,label='difference in 2011',bins=50)

plt.legend()
